# Medicare Benefits Description Generator
### LangChain + Azure OpenAI — LLM-First Approach

All processing logic lives in three plain-text prompt files — edit them without touching code:

| File | Purpose |
|---|---|
| `system_prompt.txt` | 13 formatting/substitution rules |
| `few_shot_examples.txt` | 10 worked examples covering every pattern |
| `human_template.txt` | Request structure sent per LLM call |

**Input:** `medicare_input_loadid_142.json` (single JSON file — `pbp` + `benefit_rules`)  
**Output:** `output_benefits_loadid_{load_id}.csv`

---
## Cell 1 — Load input JSON

In [ ]:
import json, os, re

INPUT_JSON_PATH = "medicare_input_loadid_142.json"

with open(INPUT_JSON_PATH, encoding="utf-8") as f:
    input_json = json.load(f)

# LoadID is in the data — no separate variable needed
load_id = str(input_json["benefit_rules"][0]["LoadID"])

print(f"Loaded: {INPUT_JSON_PATH}")
print(f"  load_id:       {load_id}")
print(f"  pbp rows:      {len(input_json['pbp']):,}")
print(f"  benefit_rules: {len(input_json['benefit_rules']):,}")


---
## Cell 2 — Install & configure Azure OpenAI

In [ ]:
# !pip install langchain langchain-openai langchain-core openai pandas   # run once

import pandas as pd

try:
    from langchain_openai import AzureChatOpenAI
except Exception:
    from langchain.chat_models import AzureChatOpenAI

from langchain_core.messages import SystemMessage, HumanMessage

AZURE_OPENAI_ENDPOINT    = os.getenv("AZURE_OPENAI_ENDPOINT",    "https://YOUR-RESOURCE.cognitiveservices.azure.com/")
AZURE_OPENAI_API_KEY     = os.getenv("AZURE_OPENAI_API_KEY",     "YOUR_API_KEY")
AZURE_OPENAI_DEPLOYMENT  = os.getenv("AZURE_OPENAI_DEPLOYMENT",  "gpt-5-mini_ng")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview")
CARRIER_PREFIX = "MOM"

print(f"Model:    {AZURE_OPENAI_DEPLOYMENT}")
print(f"Endpoint: {AZURE_OPENAI_ENDPOINT}")
print(f"Version:  {AZURE_OPENAI_API_VERSION}")


---
## Cell 3 — Load prompts from text files

All LLM instructions live in three plain-text files alongside the notebook.  
Edit them freely — no code changes needed.

```
system_prompt.txt      ← 13 rules: grouping, token substitution, formatting
few_shot_examples.txt  ← 10 worked examples covering every rule pattern
human_template.txt     ← request structure with {placeholders}
```


In [ ]:
PROMPT_DIR = "."  # folder containing the three .txt files — change if needed


def load_prompt(filename: str) -> str:
    """Read a prompt file and return its contents as a string."""
    path = os.path.join(PROMPT_DIR, filename)
    with open(path, encoding="utf-8") as f:
        return f.read()


SYSTEM_PROMPT  = load_prompt("system_prompt.txt")
FEW_SHOT       = load_prompt("few_shot_examples.txt")
HUMAN_TEMPLATE = load_prompt("human_template.txt")

# Merge few_shot directly into the system prompt so LangChain never treats
# the {{token}} placeholders in the examples as template variables.
# The human template then uses only simple {placeholders} for runtime data.
SYSTEM_PROMPT_WITH_EXAMPLES = SYSTEM_PROMPT + "\n\n" + FEW_SHOT

# Remove {few_shot} from the human template — it is now in the system prompt
HUMAN_TEMPLATE = HUMAN_TEMPLATE.replace("{few_shot}\n\n", "").replace("{few_shot}", "")

print(f"system_prompt.txt     {len(SYSTEM_PROMPT):>6,} chars")
print(f"few_shot_examples.txt {len(FEW_SHOT):>6,} chars")
print(f"human_template.txt    {len(HUMAN_TEMPLATE):>6,} chars")
print(f"combined system msg   {len(SYSTEM_PROMPT_WITH_EXAMPLES):>6,} chars")
print()
# Sanity-check that the human template contains the expected placeholders
REQUIRED = ["{carrier_prefix}", "{plan_type}", "{file_name}",
             "{n_rules}", "{rules_json}", "{n_pbp}", "{pbp_json}"]
missing = [p for p in REQUIRED if p not in HUMAN_TEMPLATE]
if missing:
    print(f"WARNING: human_template.txt is missing placeholders: {missing}")
else:
    print("All required placeholders found in human_template.txt")


---
## Cell 4 — Build LangChain chain + helpers

In [ ]:
llm = AzureChatOpenAI(
    azure_endpoint   = AZURE_OPENAI_ENDPOINT,
    api_key          = AZURE_OPENAI_API_KEY,
    azure_deployment = AZURE_OPENAI_DEPLOYMENT,
    api_version      = AZURE_OPENAI_API_VERSION,
    temperature      = 1,
    max_tokens       = 8192,
)

# Build messages directly — avoids LangChain scanning prompt files
# for {variables} and misreading JSON examples or {{tokens}} as placeholders.
def invoke_chain(template_vars: dict) -> str:
    """
    Fills HUMAN_TEMPLATE with template_vars using plain str.format_map(),
    then calls the LLM directly. No ChatPromptTemplate involved.
    """
    from langchain_core.messages import SystemMessage, HumanMessage
    human_text = HUMAN_TEMPLATE.format_map(template_vars)
    messages = [
        SystemMessage(content=SYSTEM_PROMPT_WITH_EXAMPLES),
        HumanMessage(content=human_text),
    ]
    response = llm.invoke(messages)
    return response.content


def extract_plan_meta(input_json: dict) -> dict:
    """
    Extract file_name, plan_type, and load_id from the input JSON.
    load_id is read from pbp rows (LoadID column).
    """
    pbp_rows   = input_json["pbp"]
    rules_rows = input_json["benefit_rules"]

    file_name = pbp_rows[0]["FileName"].strip() if pbp_rows else ""
    load_id   = str(pbp_rows[0]["LoadID"]).strip() if pbp_rows else ""

    plan_type = "HMO"
    for r in pbp_rows:
        if r.get("header") == "Plan Characteristics" and r.get("field") == "Plan Type":
            plan_type = r["value"].strip()
            break

    return {"file_name": file_name, "plan_type": plan_type, "load_id": load_id}


def filter_pbp_for_lookup(pbp_rows: list) -> list:
    """
    Keep only the pbp rows the LLM needs for token lookups
    (formulary exceptions, MOOP, deductibles, plan type, tier names).
    Reduces prompt size without losing any information the LLM needs.
    """
    KEEP_CATEGORIES = {
        "rx setup", "rx characteristics", "rx cost share",
        "plan level cost sharing", "tiering",
    }
    KEEP_FIELDS = {
        "what is your formulary exceptions tier?",
        "in network moop amount",
        "enter deductible amount",
        "plan type",
        "indicate each tier for which the deductible will not apply",
    }
    out = []
    for r in pbp_rows:
        cat = r.get("category", "").lower()
        fld = r.get("field",    "").lower()
        if any(c in cat for c in KEEP_CATEGORIES) or any(f in fld for f in KEEP_FIELDS):
            out.append({
                "category": r.get("category"),
                "field":    r.get("field"),
                "value":    r.get("value"),
            })
    return out


def parse_llm_output(raw: str) -> list:
    """Extract and parse the JSON array from the LLM response."""
    text = re.sub(r"^```[a-z]*\n?", "", raw.strip(), flags=re.MULTILINE)
    text = re.sub(r"\n?```$",       "", text,         flags=re.MULTILINE).strip()
    start, end = text.find("["), text.rfind("]") + 1
    if start == -1 or end == 0:
        raise ValueError("No JSON array found in LLM response")
    return json.loads(text[start:end])


print("Chain and helpers ready")


---
## Cell 5 — Run the pipeline

One LangChain call processes all benefit_rules rows and returns all output rows.


In [ ]:
meta       = extract_plan_meta(input_json)
pbp_lookup = filter_pbp_for_lookup(input_json["pbp"])

print(f"File:            {meta['file_name']}")
print(f"Load ID:         {meta['load_id']}")
print(f"Plan type:       {meta['plan_type']}")
print(f"benefit_rules:   {len(input_json['benefit_rules'])} rows")
print(f"pbp lookup rows: {len(pbp_lookup)} (filtered from {len(input_json['pbp']):,} total)")
print()
print("Calling Azure OpenAI...")

raw_response = invoke_chain({
    "carrier_prefix": CARRIER_PREFIX,
    "file_name":      meta["file_name"],
    "plan_type":      meta["plan_type"],
    "n_rules":        len(input_json["benefit_rules"]),
    "rules_json":     json.dumps(input_json["benefit_rules"], indent=2),
    "n_pbp":          len(pbp_lookup),
    "pbp_json":       json.dumps(pbp_lookup, indent=2),
})

print(f"Response received ({len(raw_response):,} chars)")


---
## Cell 6 — Parse response, export CSV, validate

In [ ]:
output_rows = parse_llm_output(raw_response)

COLS = [
    "planid", "plantypeid", "benefitid", "benefitname",
    "coveragetypeid", "coveragetypedesc",
    "serviceTypeID", "serviceTypeDesc",
    "benefitdesc", "tinyDescription",
]

df = (
    pd.DataFrame(output_rows)
    .reindex(columns=COLS)
    .sort_values(["benefitid", "serviceTypeID"])
    .reset_index(drop=True)
)

CSV_OUT = f"output_benefits_loadid_{meta['load_id']}.csv"
df.to_csv(CSV_OUT, index=False)

print(f"{len(df)} rows written to {CSV_OUT}")
df[["benefitid", "benefitname", "serviceTypeDesc", "tinyDescription"]]


In [ ]:
# ── Inspect a specific benefit in full ──────────────────────────────────────
INSPECT_ID = 1900  # change to any benefitid

for _, r in df[df["benefitid"].astype(str) == str(INSPECT_ID)].iterrows():
    print(f"[{r.benefitid}] {r.benefitname}  svc={r.serviceTypeDesc}")
    print(f"  benefitdesc:     {r.benefitdesc}")
    print(f"  tinyDescription: {r.tinyDescription}")
    print()


---
## Cell 7 — Reload prompts after editing

Edit any `.txt` file, run this cell to pick up changes without restarting the kernel.


In [ ]:
# Re-read all three prompt files — invoke_chain picks them up automatically
# since it reads SYSTEM_PROMPT_WITH_EXAMPLES and HUMAN_TEMPLATE at call time.
SYSTEM_PROMPT  = load_prompt("system_prompt.txt")
FEW_SHOT       = load_prompt("few_shot_examples.txt")
HUMAN_TEMPLATE = load_prompt("human_template.txt")

SYSTEM_PROMPT_WITH_EXAMPLES = SYSTEM_PROMPT + "\n\n" + FEW_SHOT
HUMAN_TEMPLATE = HUMAN_TEMPLATE.replace("{few_shot}\n\n", "").replace("{few_shot}", "")

print("Prompts reloaded")
print(f"  system_prompt.txt     {len(SYSTEM_PROMPT):>6,} chars")
print(f"  few_shot_examples.txt {len(FEW_SHOT):>6,} chars")
print(f"  combined system msg   {len(SYSTEM_PROMPT_WITH_EXAMPLES):>6,} chars")
